# ICT Multi-Symbol Backtest — S-006 M5

**Purpose:** Fetch real 2026 OHLCV data for all pairs in `data/ict_validate_manifest.csv`,
run the ICT backtest harness with quality filters, then produce a go/no-go validation report.

**Inputs required:**
- Colab secret `GITHUB_PAT` — read-only PAT to clone the repo (write PAT for Cell 8).

**No exchange API keys needed** — crypto data comes from Bybit public REST (no auth).

**Outputs saved to Drive:**
- `MyDrive/ict-bot-research/backtest-runs/ict_multi_YYYYMMDD.json` — raw backtest report
- `MyDrive/ict-bot-research/backtest-runs/ict_validation_report_YYYYMMDD.md` — go/no-go report

**Go criteria (M7 Phase 3):** ≥ 50 trades total · WR ≥ 55 % · avg R > 0

**Quality filters active (M4):**
- `ob_confluence_only=True` — only enter FVGs backed by an Order Block
- `disable_session_filter=True` — bypass 02–12 UTC gate for 24/7 crypto pairs

Run cells top-to-bottom. Each cell is idempotent.

In [ ]:
# Cell 1 — Install dependencies
# Note: ccxt is NOT imported against Binance (policy: testing-policy.md §"Test data sources").
# Crypto data comes from Bybit public REST via requests (no API key needed).
!pip install -q yfinance pandas requests

In [ ]:
# Cell 2 — Mount Drive and define output paths
from google.colab import drive
drive.mount('/content/drive')

import os
from datetime import datetime

RUN_DATE   = datetime.utcnow().strftime('%Y%m%d')
DRIVE_DIR  = '/content/drive/MyDrive/ict-bot-research/backtest-runs'
os.makedirs(DRIVE_DIR, exist_ok=True)

JSON_OUT   = f'{DRIVE_DIR}/ict_multi_{RUN_DATE}.json'
REPORT_OUT = f'{DRIVE_DIR}/ict_validation_report_{RUN_DATE}.md'
print(f'Outputs → {JSON_OUT}')
print(f'         {REPORT_OUT}')

In [ ]:
# Cell 3 — Clone / update repo
from google.colab import userdata
from getpass import getpass

try:
    PAT = userdata.get('GITHUB_PAT')
except Exception:
    PAT = getpass('GitHub PAT (read-only): ')

REPO = 'benbaichmankass/ict-trading-bot'
REPO_DIR = '/content/ict-trading-bot'

if not os.path.exists(REPO_DIR):
    !git clone -q https://$PAT@github.com/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q origin main

%cd {REPO_DIR}
!git log --oneline -3

In [ ]:
# Cell 4 — Fetch OHLCV data for each manifest pair
#
# Crypto pairs  → Bybit public REST (no API key, no geo-block)
# Equity pairs  → yfinance
#
# Policy: never call Binance from tests or notebooks (testing-policy.md).
#
# Data is written directly to the paths listed in the manifest so that
# backtest_ict.py --manifest can read them without any path remapping.

import requests
import yfinance as yf
import pandas as pd
from pathlib import Path
from time import sleep

MANIFEST = pd.read_csv('data/ict_validate_manifest.csv')
print(MANIFEST.to_string(index=False))

START = '2026-01-01'
END   = datetime.utcnow().strftime('%Y-%m-%d')

CRYPTO_SYMBOLS = {'BTCUSDT', 'ETHUSDT'}
EQUITY_SYMBOLS = {'SPY', 'QQQ'}

# Bybit V5 kline interval map
TF_BYBIT = {'1m': 1, '5m': 5, '15m': 15, '1h': 60, '4h': 240, '1d': 'D'}

def _bybit_klines(symbol, interval_min, start_ms, end_ms, limit=1000):
    """Single page from Bybit public kline endpoint."""
    url = 'https://api.bybit.com/v5/market/kline'
    params = {
        'category': 'linear',
        'symbol':   symbol,
        'interval': interval_min,
        'start':    start_ms,
        'end':      end_ms,
        'limit':    limit,
    }
    r = requests.get(url, params=params, timeout=15)
    r.raise_for_status()
    data = r.json()
    if data.get('retCode') != 0:
        raise RuntimeError(f"Bybit error: {data.get('retMsg')}")
    # list of [startTime, open, high, low, close, volume, turnover]
    return data['result']['list']


def fetch_crypto(symbol, timeframe, out_path):
    """Fetch via Bybit public REST and save to CSV."""
    interval = TF_BYBIT[timeframe]
    start_ms  = int(pd.Timestamp(START).timestamp() * 1000)
    end_ms    = int(pd.Timestamp(END + ' 23:59:59').timestamp() * 1000)
    all_bars  = []

    # Bybit returns newest-first; paginate backwards
    cursor_end = end_ms
    while True:
        page = _bybit_klines(symbol, interval, start_ms, cursor_end)
        if not page:
            break
        all_bars.extend(page)
        oldest_ts = int(page[-1][0])
        if oldest_ts <= start_ms:
            break
        cursor_end = oldest_ts - 1
        sleep(0.2)   # be polite

    if not all_bars:
        print(f'  WARNING: {symbol} {timeframe}: no data returned')
        return

    df = pd.DataFrame(all_bars,
                      columns=['ts', 'open', 'high', 'low', 'close', 'volume', 'turnover'])
    df['timestamp'] = pd.to_datetime(df['ts'].astype(float), unit='ms', utc=True).dt.tz_localize(None)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']].astype(
        {'open': float, 'high': float, 'low': float, 'close': float, 'volume': float})
    df = df.sort_values('timestamp').reset_index(drop=True)
    df = df[(df['timestamp'] >= START) & (df['timestamp'] <= END)]
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'  {symbol} {timeframe}: {len(df)} rows → {out_path}')


def fetch_equity(symbol, timeframe, out_path):
    """Fetch via yfinance and save to CSV."""
    ticker = yf.Ticker(symbol)
    df = ticker.history(start=START, end=END, interval=timeframe, auto_adjust=True)
    df = df.reset_index().rename(columns={
        'Datetime': 'timestamp', 'Date': 'timestamp',
        'Open': 'open', 'High': 'high', 'Low': 'low',
        'Close': 'close', 'Volume': 'volume',
    })
    df['timestamp'] = pd.to_datetime(df['timestamp']).dt.tz_localize(None)
    df = df[['timestamp', 'open', 'high', 'low', 'close', 'volume']]
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(out_path, index=False)
    print(f'  {symbol} {timeframe}: {len(df)} rows → {out_path}')


print('\nFetching OHLCV data...')
for _, row in MANIFEST.iterrows():
    sym, tf, path = row['symbol'], row['timeframe'], row['path']
    if sym in CRYPTO_SYMBOLS:
        fetch_crypto(sym, tf, path)
    elif sym in EQUITY_SYMBOLS:
        fetch_equity(sym, tf, path)
    else:
        print(f'  WARNING: unknown symbol {sym}, skipping')
print('Done.')

In [ ]:
# Cell 5 — Run the ICT backtest across all manifest pairs
#
# Quality filters (added in S-006 M4):
#   ob_confluence_only=True  → only enter FVGs backed by an Order Block
#   disable_session_filter=True → bypass 02-12 UTC gate for 24/7 crypto
import subprocess, sys, json

CONFIG = json.dumps({
    "ob_confluence_only": True,
    "disable_session_filter": True,
})

result = subprocess.run(
    [sys.executable, 'bin/backtest_ict.py',
     '--manifest', 'data/ict_validate_manifest.csv',
     '--output',   JSON_OUT,
     '--config',   CONFIG,
     '--quiet'],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(result.stdout)
if result.returncode not in (0, 1):   # 1 = some pairs failed, still ok
    print('STDERR:', result.stderr)
    raise RuntimeError(f'backtest_ict.py exited {result.returncode}')
print(f'Report written to {JSON_OUT}')

In [ ]:
# Cell 6 — Run the analyzer and produce the go/no-go validation report
result = subprocess.run(
    [sys.executable, 'bin/analyze_ict_results.py',
     '--input',      JSON_OUT,
     '--output',     REPORT_OUT,
     '--min-trades', '50',
     '--min-wr',     '55'],
    capture_output=True, text=True,
    env={**os.environ, 'PYTHONPATH': REPO_DIR},
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError(f'analyze_ict_results.py exited {result.returncode}')

In [ ]:
# Cell 7 — Display the validation report inline
from IPython.display import Markdown, display
display(Markdown(open(REPORT_OUT).read()))

In [ ]:
# Cell 8 — (Optional) Copy validation report into the repo and commit
#
# Run this cell only if the go/no-go report is ready to be checked in.
# Requires a PAT with write access (push permission).

import shutil

REPO_REPORT = f'{REPO_DIR}/docs/sprint-plans/ict-validation-report.md'
shutil.copy(REPORT_OUT, REPO_REPORT)

!git -C {REPO_DIR} config user.email 'colab@ict-bot'
!git -C {REPO_DIR} config user.name  'Colab'
!git -C {REPO_DIR} add docs/sprint-plans/ict-validation-report.md
!git -C {REPO_DIR} commit -m 'feat(s006-m3): ICT multi-symbol validation report {RUN_DATE}'
!git -C {REPO_DIR} push https://$PAT@github.com/{REPO}.git main
print('Committed and pushed.')

## Handoff to Claude

After running all cells, copy the following back into the Claude Code session:

1. The printed **Verdict** line from Cell 6 (`GO` / `NO-GO` + trade count + WR + avg R).
2. The full contents of `REPORT_OUT` (displayed in Cell 7), or the Drive path.
3. Any error output from Cells 4–6.

Claude will then:
- If **GO**: open a PR to wire the ICT strategy into the live pipeline (S-006 M6).
- If **NO-GO**: document the shortfall and recommend next parameter-tuning steps.